# 2. Off-the-shelf baselines (Section 3.1, Table 1)

1. `embed.py` embeds every validation and test paper with each model and caches one `.npy` per model and pooling in `BASE/embeddings/` (finished models are skipped when re-run).
2. `evaluate.py` computes retrieval, clustering and pair metrics with cluster-bootstrap CIs.

GPU runtime. Run notebook 1 first if the cross-encoder should appear in the table.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, json, os

REPO = 'https://github.com/GITHUB_ID/pathway-supervised-embedder.git'     # <- this repository
CODE = '/content/pathway-supervised-embedder'
BASE = '/content/drive/MyDrive/kegg-bge'               # Drive folder: data/splits/ and all results
DATA = '/content/data/splits'                          # local copy of the benchmark (faster reads)

if os.path.isdir(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q {REPO} {CODE}
S = f'{CODE}/scripts'
!mkdir -p /content/data {BASE}
!cp -r {BASE}/data/splits /content/data/
def latest(d): return sorted(glob.glob(f'{d}/run-*'))[-1]

In [ ]:
EMB  = f'{BASE}/embeddings'
EVAL = f'{BASE}/eval_baselines'
CROSS = latest(f'{BASE}/result_crossencoder') if glob.glob(f'{BASE}/result_crossencoder/run-*') else None
print('cross-encoder run:', CROSS)

## Embeddings: abstract only (same input as our model)
Transformer models save CLS, mean and final-token pooling; the pooling is chosen on the validation split in `evaluate.py`.

In [ ]:
ours = f'--ours {CROSS}/best' if CROSS else ''     # 'ours' = embeddings taken from the cross-encoder (Section 2.5)
!python {S}/embed.py --data-dir {DATA} --out-dir {EMB} --bf16 {ours} \
    --models longformer biobert scibert pubmedbert specter medcpt bge tfidf random

## Embeddings: each model's recommended input (title + abstract)
SPECTER and MedCPT were trained on titles and abstracts; saved as `specter_native`, `medcpt_native` (T+A in the paper).

In [ ]:
!python {S}/embed.py --data-dir {DATA} --out-dir {EMB} --bf16 --input native --models specter medcpt

## SPECTER2 (run last)
`adapters` may change the installed transformers version, so run this after the other models.

In [ ]:
!pip -q install adapters
!python {S}/embed.py --data-dir {DATA} --out-dir {EMB} --bf16 --models specter2
!python {S}/embed.py --data-dir {DATA} --out-dir {EMB} --bf16 --input native --models specter2

## Evaluation

In [ ]:
!pip -q install tabulate
pair = f'--pair-preds cross_encoder={CROSS}/test_predictions.jsonl' if CROSS else ''
!python {S}/evaluate.py --data-dir {DATA} --emb-dir {EMB} --out-dir {EVAL} --reference bge {pair}
from IPython.display import Markdown
Markdown(open(f'{EVAL}/report_test.md').read())